In [2]:
import cv2
import numpy as np
from ultralytics import YOLO

# 1. Load your model and image
model = YOLO("/home/user/SolarVortex/CellExtraction/YOLO/YOLOv11-seg/runs/segment/YOLO11_SEG_Benchmark/YOLO11n-Seg/weights/best.pt")
image_path = r"/home/user/SolarVortex/CellExtraction/YOLO/YOLOv8-OBB/CellDetection-9/train/images/ARTS_00009_r4_c1_png.rf.3e3a1dd3cce49866ca00ae66aa9288dc.jpg"
img = cv2.imread(image_path)

results = model(image_path)

for result in results:
    if result.masks is None:
        print("No solar cells detected.")
        continue

    # Get raw masks and scale them to original image size
    masks = result.masks.data.cpu().numpy()
    
    for i, mask in enumerate(masks):
        # Resize mask to original image size
        mask_resized = cv2.resize(mask, (img.shape[1], img.shape[0]), interpolation=cv2.INTER_NEAREST)
        binary_mask = (mask_resized > 0.5).astype(np.uint8) * 255
        
        # Find the contours/coordinates of the polygon mask
        contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not contours:
            continue
            
        # Get the largest contour for this specific cell
        c = max(contours, key=cv2.contourArea)
        
        # Find the minimum area bounding box (handles rotation/tilted rectangles)
        rect = cv2.minAreaRect(c)
        box_points = cv2.boxPoints(rect)
        box_points = np.int64(box_points)
        
        # Get width and height of the detected cell
        width = int(rect[1][0])
        height = int(rect[1][1])
        
        # Get transformation matrix to rotate the image upright
        src_pts = box_points.astype("float32")
        dst_pts = np.array([
            [0, height - 1],
            [0, 0],
            [width - 1, 0],
            [width - 1, height - 1]
        ], dtype="float32")
        
        M = cv2.getPerspectiveTransform(src_pts, dst_pts)
        
        # Warp the image to straighten the diamond/rhombus into a clean rectangle
        warped_cell = cv2.warpPerspective(img, M, (width, height))
        
        # --- VISUALIZATION WINDOWS ---
        # Window 1: Show original image with the calculated oriented bounding box overlay
        img_with_box = img.copy()
        cv2.drawContours(img_with_box, [box_points], 0, (0, 255, 0), 2)
        cv2.imshow(f"Detected Tilted Cell {i}", cv2.resize(img_with_box, (800, 600)))
        
        # Window 2: Show the perfectly straightened rectangular crop
        cv2.imshow(f"Straightened Crop {i}", warped_cell)
        
        print(f"Visualizing Cell {i}. Press any key in the image window to see the next cell...")
        cv2.waitKey(0)  # Wait for a key press to show the next detected cell

cv2.destroyAllWindows()


image 1/1 /home/user/SolarVortex/CellExtraction/YOLO/YOLOv8-OBB/CellDetection-9/train/images/ARTS_00009_r4_c1_png.rf.3e3a1dd3cce49866ca00ae66aa9288dc.jpg: 640x640 1 cell, 4.1ms
Speed: 1.0ms preprocess, 4.1ms inference, 1.9ms postprocess per image at shape (1, 3, 640, 640)
Visualizing Cell 0. Press any key in the image window to see the next cell...


QFontDatabase: Cannot find font directory /home/user/anaconda3/envs/raj/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/user/anaconda3/envs/raj/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/user/anaconda3/envs/raj/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/user/anaconda3/envs/raj/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /h

In [1]:
import cv2
import numpy as np
from ultralytics import YOLO

# 1. Load the different model architectures
models = {
    "YOLOv8-Normal (Standard Axis-Aligned Box)": YOLO("/home/user/SolarVortex/CellExtraction/YOLO/YOLOv8/runs/detect/benchmark_results/yolov8l/weights/best.pt"),
    "YOLOv8-OBB (Oriented Tilted Box)": YOLO("/home/user/SolarVortex/CellExtraction/YOLO/YOLOv8-obb-2/runs/obb/OBB_Benchmark/yolov8l-obb/weights/best.pt"),
    "YOLOv11-Seg (Your Custom Polygon Mask Model)": YOLO("/home/user/SolarVortex/CellExtraction/YOLO/YOLOv11-seg/runs/segment/YOLO11_SEG_Benchmark/YOLO11n-Seg/weights/best.pt")
}

image_path = "/home/user/SolarVortex/CellExtraction/YOLO/YOLOv8-OBB/CellDetection-9/train/images/ARTS_00009_r4_c1_png.rf.3e3a1dd3cce49866ca00ae66aa9288dc.jpg"

for model_name, model in models.items():
    # Load a fresh copy of the image for clean annotations
    canvas = cv2.imread(image_path)
    results = model(image_path)
    result = results[0]
    
    print(f"\nVisualizing annotation types for: {model_name}")

    # CASE A: Standard Horizontal Bounding Boxes (Normal YOLOv8)
    if model_name == "YOLOv8-Normal (Standard Axis-Aligned Box)" and result.boxes is not None:
        boxes = result.boxes.xyxy.cpu().numpy()
        for box in boxes:
            x1, y1, x2, y2 = map(int, box)
            # Blue, flat standard rectangle
            cv2.rectangle(canvas, (x1, y1), (x2, y2), (255, 0, 0), 2)
            cv2.putText(canvas, "Standard Box", (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2)

    # CASE B: Oriented Bounding Boxes (YOLOv8-OBB)
    elif model_name == "YOLOv8-OBB (Oriented Tilted Box)" and hasattr(result, 'obb') and result.obb is not None:
        obb_boxes = result.obb.xyxyxyxy.cpu().numpy()
        for box_points in obb_boxes:
            box_points = np.int64(box_points)
            # Yellow, naturally tilted rectangle mapping the rotation
            cv2.polylines(canvas, [box_points], isClosed=True, color=(0, 255, 255), thickness=2)
            cv2.putText(canvas, "OBB", (box_points[0][0], box_points[0][1] - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 2)

    # CASE C: Bounding Box derived from a Polygon Segmentation Mask (YOLOv11-Seg)
    elif model_name == "YOLOv11-Seg (Your Custom Polygon Mask Model)" and result.masks is not None:
        masks = result.masks.data.cpu().numpy()
        for mask in masks:
            mask_resized = cv2.resize(mask, (canvas.shape[1], canvas.shape[0]), interpolation=cv2.INTER_NEAREST)
            binary_mask = (mask_resized > 0.5).astype(np.uint8) * 255
            
            contours, _ = cv2.findContours(binary_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            if contours:
                c = max(contours, key=cv2.contourArea)
                rect = cv2.minAreaRect(c)
                box_points = np.int64(cv2.boxPoints(rect))
                # Green, minimum-area rectangle hugging the polygon shape edge
                cv2.drawContours(canvas, [box_points], 0, (0, 255, 0), 2)
                cv2.putText(canvas, "Mask MinArea Box", (box_points[1][0], box_points[1][1] - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    # Display the result for the model
    window_title = f"Annotation Map: {model_name}"
    cv2.imshow(window_title, cv2.resize(canvas, (900, 650)))
    print("Click on the window and press ANY key to see the next model's annotation layout...")
    cv2.waitKey(0)
    cv2.destroyWindow(window_title)

cv2.destroyAllWindows()


image 1/1 /home/user/SolarVortex/CellExtraction/YOLO/YOLOv8-OBB/CellDetection-9/train/images/ARTS_00009_r4_c1_png.rf.3e3a1dd3cce49866ca00ae66aa9288dc.jpg: 640x640 1 cell, 6.0ms
Speed: 0.9ms preprocess, 6.0ms inference, 14.3ms postprocess per image at shape (1, 3, 640, 640)

Visualizing annotation types for: YOLOv8-Normal (Standard Axis-Aligned Box)
Click on the window and press ANY key to see the next model's annotation layout...


QFontDatabase: Cannot find font directory /home/user/anaconda3/envs/raj/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/user/anaconda3/envs/raj/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/user/anaconda3/envs/raj/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /home/user/anaconda3/envs/raj/lib/python3.12/site-packages/cv2/qt/fonts.
Note that Qt no longer ships fonts. Deploy some (from https://dejavu-fonts.github.io/ for example) or switch to fontconfig.
QFontDatabase: Cannot find font directory /h


image 1/1 /home/user/SolarVortex/CellExtraction/YOLO/YOLOv8-OBB/CellDetection-9/train/images/ARTS_00009_r4_c1_png.rf.3e3a1dd3cce49866ca00ae66aa9288dc.jpg: 640x640 1 cell, 6.2ms
Speed: 0.7ms preprocess, 6.2ms inference, 71.4ms postprocess per image at shape (1, 3, 640, 640)

Visualizing annotation types for: YOLOv8-OBB (Oriented Tilted Box)
Click on the window and press ANY key to see the next model's annotation layout...

image 1/1 /home/user/SolarVortex/CellExtraction/YOLO/YOLOv8-OBB/CellDetection-9/train/images/ARTS_00009_r4_c1_png.rf.3e3a1dd3cce49866ca00ae66aa9288dc.jpg: 640x640 1 cell, 4.2ms
Speed: 1.1ms preprocess, 4.2ms inference, 2.1ms postprocess per image at shape (1, 3, 640, 640)

Visualizing annotation types for: YOLOv11-Seg (Your Custom Polygon Mask Model)
Click on the window and press ANY key to see the next model's annotation layout...
